# 🔐 Giorno 3 - Lab 3: Pattern Human Approval con LangChain

Benvenuti al terzo laboratorio del **Giorno 3**!

Nel Lab 2 abbiamo costruito un agente che decide autonomamente usando `AgentExecutor`.
Ora aggiungiamo il **pattern Human Approval** usando le funzionalità di LangChain:
- `BaseCallbackHandler` — intercetta gli eventi dell'agente (tool calls, risposte) per triggering il checkpoint
- `@tool` per `request_purchase_approval` — il tool di escalation che attiva il checkpoint
- Tracciamento con `intermediate_steps` — per estrarre la decisione senza parsing manuale

---
### 🎯 Obiettivo
Aggiungere al flow del Lab 2 un componente di approvazione che:
- Si attiva quando l'agente invoca `request_purchase_approval` (decisione di escalation)
- **Mostra all'operatore** il riepilogo e chiede conferma esplicita
- **Registra** la decisione umana in un audit trail

---
### 🗺️ Schema
```
input → AgentExecutor
          ├─ @tool get_employee_budget
          ├─ @tool search_product_catalog
          └─ @tool request_purchase_approval
                       │
               HumanApprovalCallback
               (intercetta la call,
                chiede conferma)
                       │
               Esegue solo se approvato
                       │
output ← AgentExecutor (risposta finale + audit)
```

## 1. Setup — Riutilizziamo i Tool del Lab 2

In [2]:
import json
import datetime
from pathlib import Path
from typing import Any, Union

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks import BaseCallbackHandler
try:
    from langchain.agents import AgentExecutor, create_tool_calling_agent
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

DATA_DIR = Path("data")
llm      = ChatOllama(model="llama3.2", temperature=0)
print("✅ Setup completato.")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Helper privati (non esposti come @tool — usati internamente dall'orchestratore)
# ══════════════════════════════════════════════════════════════════════════

def _find_product(query: str) -> dict | None:
    """Ritorna il prodotto con il punteggio di match più alto, o None."""
    products = json.loads((DATA_DIR / "products.json").read_text())
    q = query.lower().strip()
    # 1. Match esatto sulla stringa intera
    for pid, p in products.items():
        if q in p["name"].lower() or q == p["category"].lower():
            return {"pid": pid, **p}
    # 2. Fallback score-based: token ≥ 4 char, ignora stopword
    stopwords = {"il","la","un","una","del","per","di","i","le","gli",
                 "sono","quali","che","nel","al","the","for","and","or"}
    terms = [t for t in q.split() if t not in stopwords and len(t) >= 4]
    if not terms:
        return None
    scores = {}
    for pid, p in products.items():
        score = sum(1 for t in terms
                    if t in p["name"].lower() or t in p["category"].lower())
        if score > 0:
            scores[pid] = score
    if not scores:
        return None
    best_pid = max(scores, key=scores.__getitem__)
    return {"pid": best_pid, **products[best_pid]}


def _apply_policy(emp: dict, product: dict) -> str | None:
    """Applica le regole della policy. Ritorna la motivazione di escalation,
    oppure None se l'acquisto è approvabile automaticamente."""
    dept   = emp["department"]
    budget = emp["budget_residuo"]
    price  = product["price_eur"]
    cat    = product["category"]
    is_tech = dept in ("Engineering", "Tech", "Data Science")
    if cat in ("monitor", "accessory") and price > 300 and not is_tech:
        return f"{cat.capitalize()} {price}€ > 300€ per ruolo non-Engineering ({dept})."
    if price > budget:
        return f"Prezzo {price}€ supera il budget residuo {budget}€."
    fascia = 1800 if is_tech else 1000
    if cat == "laptop" and price > fascia:
        return f"Laptop {price}€ supera la fascia {fascia}€ per {dept}."
    return None


# ══════════════════════════════════════════════════════════════════════════
# Tool separati — visibili nel notebook per gli obiettivi didattici
# ══════════════════════════════════════════════════════════════════════════

@tool
def get_employee_budget(employee_id: str) -> dict:
    """Recupera budget residuo, ruolo, dipartimento e progetto attivo di un dipendente.
    Il parametro employee_id è il BADGE ID alfanumerico (es. 'E4521'), NON il nome."""
    employees = json.loads((DATA_DIR / "employees.json").read_text())
    if employee_id not in employees:
        return {"status": "NOT_FOUND", "message": f"Dipendente '{employee_id}' non trovato."}
    emp = employees[employee_id]
    return {"status": "OK", "employee_id": employee_id, "name": emp["name"],
            "department": emp["department"], "role": emp["role"],
            "budget_residuo_eur": emp["budget_residuo"],
            "active_project": emp.get("active_project"), "manager_email": emp["manager_email"]}


@tool
def search_product_catalog(query: str) -> dict:
    """Cerca prodotti nel catalogo hardware aziendale per nome o categoria.
    Esempi di query: 'Dell UltraSharp', 'monitor', 'ThinkPad', 'laptop'."""
    product = _find_product(query)
    if product is None:
        return {"status": "NOT_FOUND", "message": f"Nessun prodotto trovato per '{query}'."}
    pid = product.pop("pid")
    return {"status": "OK", "risultati": {pid: product}, "totale": 1}


# ══════════════════════════════════════════════════════════════════════════
# Orchestratore Python: chiama i tool sopra come funzioni e gestisce il
# ticket. È l'UNICO tool esposto all'agente LLM.
# ══════════════════════════════════════════════════════════════════════════

@tool
def evaluate_purchase_request(employee_id: str, product_query: str) -> dict:
    """Valuta una richiesta di acquisto: recupera i dati del dipendente,
    cerca il prodotto nel catalogo, applica la policy aziendale e — se necessario —
    crea il ticket di approvazione. Restituisce la decisione finale.
    employee_id: badge ID alfanumerico es. 'E1082' (NON il nome).
    product_query: nome o descrizione del prodotto."""
    # 1. Dati dipendente (chiamata Python diretta a get_employee_budget)
    emp_result = get_employee_budget.func(employee_id)
    if emp_result.get("status") == "NOT_FOUND":
        return {"decision": "REJECTED", "reason": emp_result["message"]}
    emp = json.loads((DATA_DIR / "employees.json").read_text())[employee_id]
    # 2. Prodotto (chiamata Python diretta a _find_product)
    product = _find_product(product_query)
    if product is None:
        return {"decision": "REJECTED",
                "reason": f"Prodotto '{product_query}' non trovato nel catalogo."}
    product.pop("pid", None)
    # 3. Regole policy
    escalation_reason = _apply_policy(emp, product)
    prod_name = product["name"]
    price     = product["price_eur"]
    # 4. Se escalation: crea il ticket (chiama request_purchase_approval come funzione)
    if escalation_reason:
        ticket_result = request_purchase_approval.func(
            employee_id  = employee_id,
            product_name = prod_name,
            price_eur    = price,
            reason       = escalation_reason,
            emp_name     = emp["name"],
            active_project = emp.get("active_project", ""),
            manager_email  = emp["manager_email"],
        )
        return {
            "decision":     "NEEDS_HUMAN_APPROVAL",
            "ticket_id":    ticket_result["ticket_id"],
            "product_name": prod_name,
            "price_eur":    price,
            "reason":       escalation_reason,
            "message":      ticket_result["message"],
        }
    # 5. Approvazione automatica
    return {"decision": "APPROVED",
            "reason": f"Prodotto {price}€ rientra nella fascia e nel budget residuo.",
            "employee_name": emp["name"], "product_name": prod_name, "price_eur": price}


print("✅ Tool base pronti.")


✅ Tool base pronti.


## 2. Il Tool `request_purchase_approval`

Il nuovo tool è quello che **scatena il checkpoint** Human Approval.
L'agente lo chiama quando decide che la richiesta richiede escalation al manager.

In LangChain, il momento naturale per intercettare questo evento è il **callback** — lo implementiamo nella sezione 3.

In [ ]:
# Registro globale dei ticket (simulazione DB in memoria)
TICKET_STORE: list[dict] = []

@tool
def request_purchase_approval(
    employee_id: str,
    product_name: str,
    price_eur: Union[float, int, str, None] = 0.0,
    reason: str = "",
    emp_name: str = "",
    active_project: str = "",
    manager_email: str = "manager@azienda.it",
) -> dict:
    """Invia una richiesta formale di approvazione al manager (crea un ticket di acquisto).
    Usare per qualsiasi richiesta che superi il budget residuo, sia fuori fascia standard del ruolo,
    o sia un monitor/accessorio specialistico > 300 EUR per ruoli non-Engineering.
    Parametri obbligatori: employee_id, product_name, price_eur, reason."""
    try:
        p_val = float(price_eur) if price_eur is not None else 0.0
    except (ValueError, TypeError):
        p_val = 0.0
    ticket_id = f"TKT-{len(TICKET_STORE) + 1:04d}"
    ticket = {
        "ticket_id":      ticket_id,
        "employee_id":    employee_id,
        "employee_name":  emp_name or employee_id,
        "product":        product_name,
        "product_name":   product_name,
        "price_eur":      p_val,
        "reason":         str(reason or ""),
        "status":         "PENDING_APPROVAL",
        "created_at":     datetime.datetime.now().isoformat(),
        "manager_email":  manager_email,
        "active_project": active_project,
    }
    TICKET_STORE.append(ticket)
    print(f"\n  🎫 Ticket creato: {ticket_id} | {employee_id} | {product_name} | {p_val}€")
    return {
        "status":    "PENDING_HUMAN_APPROVAL",
        "ticket_id": ticket_id,
        "message":   f"Richiesta registrata con ID {ticket_id}. In attesa di approvazione dal manager."
    }

print("✅ Tool request_purchase_approval definito.")


✅ Tool request_purchase_approval definito.


## 3. Il Human Approval Callback

LangChain offre i **Callbacks** come meccanismo di osservabilità e controllo:
- `on_tool_start` — chiamato prima dell'esecuzione di ogni tool
- `on_tool_end` — chiamato dopo l'esecuzione di ogni tool
- `on_agent_finish` — chiamato quando l'agente ha la risposta finale

Usiamo `on_tool_end` per intercettare la risposta di `request_purchase_approval` e chiedere la conferma umana.

> **Nota**: In un sistema di produzione (LangGraph con `interrupt()`), il flow si sospende davvero e riprende dopo la risposta umana. Qui, dato che siamo in un notebook e non usiamo LangGraph, implementiamo il checkpoint in modo sincrono: `on_tool_end` blocca l'esecuzione con `input()` finché l'operatore non risponde.

In [ ]:
class HumanApprovalCallback(BaseCallbackHandler):
    """
    LangChain Callback usato come audit logger.
    Registra tutte le tool call dell'agente per osservabilità.

    NOTA: in un notebook Jupyter, `input()` all'interno di un callback
    non viene instradato verso la cella — il checkpoint interattivo viene
    quindi gestito direttamente nella cella di test tramite
    `human_approval_checkpoint()` dopo che l'agente ha completato il run.
    """

    def __init__(self, auto_mode: bool = False):
        self.auto_mode      = auto_mode
        self.audit_log      = []
        self.last_decision: str = None

    def on_tool_end(
        self,
        output: Any,
        *,
        run_id: Any = None,
        parent_run_id: Any = None,
        **kwargs: Any,
    ) -> None:
        """Registra nell'audit log quando il tool segnala un ticket pending."""
        if "PENDING_HUMAN_APPROVAL" not in str(output):
            return
        ticket = TICKET_STORE[-1] if TICKET_STORE else {}
        self.audit_log.append({
            "event":     "ticket_created",
            "ticket_id": ticket.get("ticket_id"),
            "timestamp": datetime.datetime.now().isoformat()
        })


def human_approval_checkpoint(approval_cb: HumanApprovalCallback) -> None:
    """
    Checkpoint Human Approval da chiamare nella cella di test
    DOPO executor.invoke().

    Cerca i ticket PENDING_APPROVAL nel TICKET_STORE, mostra il riepilogo
    e chiede la conferma dell'operatore (o simula in auto_mode).
    """
    pending = [t for t in TICKET_STORE if t.get("status") == "PENDING_APPROVAL"]
    if not pending:
        return  # nessuna escalation → niente da approvare

    for ticket in pending:
        print("\n" + "═" * 60)
        print("  🔐 HUMAN APPROVAL CHECKPOINT")
        print("═" * 60)
        print(f"  Dipendente:  {ticket.get('employee_name')} ({ticket.get('employee_id')})")
        print(f"  Prodotto:    {ticket.get('product')}")
        print(f"  Prezzo:      {ticket.get('price_eur')} EUR")
        print(f"  Motivazione: {ticket.get('reason')}")
        print(f"  Progetto:    {ticket.get('active_project')}")
        print(f"  Manager:     {ticket.get('manager_email')}")
        print(f"  Ticket ID:   {ticket.get('ticket_id')}")
        print("-" * 60)

        if approval_cb.auto_mode:
            print("  [AUTO_MODE] Approvazione simulata: SÌ")
            approved      = True
            operator_note = "Approvato via auto_mode (test)"
        else:
            scelta = input("\n  ❓ Approvi il ticket? [s/n]: ").strip().lower()
            approved = scelta in ("s", "y", "si", "yes")
            if approved:
                print("  ✅ Operatore ha APPROVATO.")
                operator_note = input("  Nota (invio per saltare): ").strip() or "Approvato"
            else:
                print("  ❌ Operatore ha RIFIUTATO.")
                operator_note = input("  Motivazione del rifiuto: ").strip() or "Rifiutato dall'operatore"

        approval_cb.last_decision = "approved" if approved else "rejected"
        ticket["status"]        = "APPROVED" if approved else "REJECTED_BY_OPERATOR"
        ticket["operator_note"] = operator_note
        ticket["decided_at"]    = datetime.datetime.now().isoformat()

        audit_record = {
            "ticket_id":     ticket.get("ticket_id"),
            "approved":      approved,
            "operator_note": operator_note,
            "timestamp":     datetime.datetime.now().isoformat()
        }
        approval_cb.audit_log.append(audit_record)
        print(f"\n  📁 Audit record salvato: {ticket['ticket_id']} → {'APPROVATO' if approved else 'RIFIUTATO'}")
        print("═" * 60)


print("✅ HumanApprovalCallback e human_approval_checkpoint definiti.")

✅ HumanApprovalCallback e human_approval_checkpoint definiti.


## 4. System Prompt V2 con Tool Approvazione

In [ ]:
SYSTEM_PROMPT_V2 = """Sei l'Assistente Aziendale per gli Acquisti Hardware.

Hai un solo tool: 'evaluate_purchase_request'.
Chiamalo SEMPRE con il badge ID del dipendente e il nome del prodotto.
Il tool applica la policy, crea il ticket se necessario e restituisce la decisione.
Dopo aver ricevuto la risposta del tool, comunica all'utente l'esito in italiano.
"""

prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_V2),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✅ Prompt V2 pronto.")


✅ Prompt V2 pronto.


## 5. Assemblaggio del Flow con Human Approval

In [ ]:
def crea_executor_con_approval(auto_mode: bool = True):
    """Crea un AgentExecutor con il callback HumanApproval integrato."""
    tools_v2 = [evaluate_purchase_request]
    agent_v2 = create_tool_calling_agent(llm, tools_v2, prompt_v2)

    approval_cb = HumanApprovalCallback(auto_mode=auto_mode)

    executor_v2 = AgentExecutor(
        agent    = agent_v2,
        tools    = tools_v2,
        verbose  = True,
        callbacks = [approval_cb],       # ← il callback è qui
        max_iterations = 10,
        handle_parsing_errors = True,
        return_intermediate_steps = True,
    )
    return executor_v2, approval_cb

print("✅ Factory del flow con Human Approval pronta.")

✅ Factory del flow con Human Approval pronta.


## 6. Test: Escalation con Human Approval

Giulia Verdi chiede il monitor 4K → escalation → il callback intercetta e chiede conferma.

> `auto_mode=True` simula la conferma automatica. Cambia in `False` per la versione interattiva.

In [ ]:
# Resetta il ticket store
TICKET_STORE.clear()

executor_v2, approval_cb = crea_executor_con_approval(auto_mode=False)

print("=" * 60)
print("  TEST: Escalation Giulia Verdi + Human Approval")
print("=" * 60)

result_giulia = executor_v2.invoke({"input": (
    "Salve, sono Giulia Verdi (badge E1082) del team Marketing. "
    "Per la Brand Campaign Q2 ho bisogno del monitor 4K Dell UltraSharp. "
    "Posso procedere all'acquisto?"
)})

print("\n📤 RISPOSTA AGENTE:", result_giulia["output"])

# ── Human Approval Checkpoint ──────────────────────────────────────────────
# Viene chiamato qui, nella cella, dopo che l'agente ha completato il run.
# input() in questa posizione è gestito correttamente da Jupyter.
human_approval_checkpoint(approval_cb)

  TEST: Escalation Giulia Verdi + Human Approval


> Entering new AgentExecutor chain...

Invoking: `evaluate_purchase_request` with `{'product_query': 'monitor 4K Dell UltraSharp', 'employee_id': 'E1082'}`



  🎫 Ticket creato: TKT-0001 | E1082 | Dell UltraSharp 27 4K (U2723QE) | 520.0€
{'decision': 'NEEDS_HUMAN_APPROVAL', 'ticket_id': 'TKT-0001', 'product_name': 'Dell UltraSharp 27 4K (U2723QE)', 'price_eur': 520, 'reason': 'Monitor 520€ > 300€ per ruolo non-Engineering (Marketing).', 'message': 'Richiesta registrata con ID TKT-0001. In attesa di approvazione dal manager.'}Ciao Giulia Verdi,

La tua richiesta di acquisto del monitor 4K Dell UltraSharp è stata registrata con l'ID TKT-0001. Tuttavia, il sistema ha deciso che richiede l'approvazione del tuo manager a causa del prezzo superiore al limite di 300€ per ruolo non-Engineering.

Ti invio il ticket con l'ID TKT-0001, in modo che tu possa condividerlo con il tuo manager e ottenere la sua approvazione. Una volta approvata, potrai

In [ ]:
# Approvazione automatica — NON deve attivare il callback
# TICKET_STORE non viene resettato qui: vogliamo che il ticket di Giulia Verdi
# rimanga visibile nell'ispezione della sezione 7.
executor_v2b, cb2 = crea_executor_con_approval(auto_mode=True)

print("=" * 60)
print("  TEST: Approvazione automatica Mario Rossi")
print("=" * 60)

result_mario = executor_v2b.invoke({"input": (
    "Ciao! Sono Mario Rossi, badge E4521. "
    "Ho bisogno del ThinkPad P16s per il Project Delta."
)})

print("\n📤 RISPOSTA FINALE:", result_mario["output"])
print("\n  Callback attivato:", len(cb2.audit_log) > 0)
print("  (Per approvazione automatica il callback NON deve essere attivato)")

  TEST: Approvazione automatica Mario Rossi


> Entering new AgentExecutor chain...

Invoking: `evaluate_purchase_request` with `{'employee_id': 'E4521', 'product_query': 'ThinkPad P16s per il Project Delta'}`


{'decision': 'APPROVED', 'reason': 'Prodotto 1650€ rientra nella fascia e nel budget residuo.', 'employee_name': 'Mario Rossi', 'product_name': 'ThinkPad P16s Gen 2 / MacBook Pro M3', 'price_eur': 1650}Ciao Mario Rossi, badge E4521!

Hai richiesto il ThinkPad P16s per il Project Delta e il nostro tool ha valutato la tua richiesta.

La decisione è stata la seguente: **APPROVED**.

Il motivo è che il ThinkPad P16s, con un prezzo di 1650€, rientra nella fascia di budget residuo per il tuo progetto. 

Il prodotto è stato approvato per il tuo utilizzo.

Se hai altre domande o richieste, non esitare a chiedere!

> Finished chain.

📤 RISPOSTA FINALE: Ciao Mario Rossi, badge E4521!

Hai richiesto il ThinkPad P16s per il Project Delta e il nostro tool ha valutato la tua richiesta.

La d

## 7. Ispezione dei Passaggi Intermedi e dell'Audit Trail

In [ ]:
print("=" * 60)
print("  AUDIT TRAIL")
print("=" * 60)
if approval_cb.audit_log:
    for record in approval_cb.audit_log:
        print(json.dumps(record, indent=2, ensure_ascii=False))
else:
    print("  Nessun record di approvazione umana (tutti automatici).")

print("\n  TICKET STORE:")
for t in TICKET_STORE:
    print(f"  [{t['ticket_id']}] {t['status']} — {t['employee_name']} / {t['product']}")

  AUDIT TRAIL
{
  "ticket_id": "TKT-0001",
  "approved": false,
  "operator_note": "boh",
  "timestamp": "2026-10-01T17:07:15.771794"
}

  TICKET STORE:
  [TKT-0001] REJECTED_BY_OPERATOR — Giulia Verdi / Dell UltraSharp 27 4K (U2723QE)


In [ ]:
# Ispezione degli intermediate_steps
print("=" * 60)
print("  PASSAGGI INTERMEDI (intermediate_steps) — Giulia Verdi")
print("=" * 60)
for i, (action, obs) in enumerate(result_giulia["intermediate_steps"], 1):
    print(f"  Step {i}: tool='{action.tool}'")
    print(f"           args={action.tool_input}")
    obs_str = json.dumps(obs, ensure_ascii=False) if isinstance(obs, (dict, list)) else str(obs)
    print(f"           obs= {obs_str[:120]}")

  PASSAGGI INTERMEDI (intermediate_steps) — Giulia Verdi
  Step 1: tool='evaluate_purchase_request'
           args={'product_query': 'monitor 4K Dell UltraSharp', 'employee_id': 'E1082'}
           obs= {"decision": "NEEDS_HUMAN_APPROVAL", "ticket_id": "TKT-0001", "product_name": "Dell UltraSharp 27 4K (U2723QE)", "price_


## 8. 💡 Esercizi Pratici

### Esercizio A — Test Interattivo
Esegui il flow con `auto_mode=False` e prova a **rifiutare** la richiesta di Giulia Verdi.
Osserva come cambia la risposta finale e l'audit record.

In [ ]:
# ESERCIZIO A — Decommenta e prova il rifiuto interattivo
# TICKET_STORE.clear()
# exec_interattivo, cb_interattivo = crea_executor_con_approval(auto_mode=False)
# result_interattivo = exec_interattivo.invoke({"input": (
#     "Salve, sono Giulia Verdi (badge E1082). Voglio il monitor 4K Dell UltraSharp."
# )})
# print(result_interattivo["output"])
# print("\nAudit:", cb_interattivo.audit_log)
print("💡 Decommenta il blocco sopra per il test interattivo.")

💡 Decommenta il blocco sopra per il test interattivo.


### Esercizio B — Secondo Callback: Logger
Crea un secondo callback `ToolLogger` che registra tutte le tool call eseguite (nome, argomenti, durata) senza interrompere il flow.
Aggiungilo all'`AgentExecutor` insieme a `HumanApprovalCallback`.

In [ ]:
# ESERCIZIO B — ToolLogger callback
import time

class ToolLogger(BaseCallbackHandler):
    def __init__(self):
        self.log = []
        self._start_time: float = 0.0
        self._current_tool: str = ""

    def on_tool_start(self, serialized: dict, input_str: str, **kwargs: Any) -> None:
        # TODO: registra nome del tool e timestamp di inizio
        self._start_time  = time.time()
        self._current_tool = serialized.get("name", "unknown")

    def on_tool_end(self, output: Any, **kwargs: Any) -> None:
        # TODO: calcola la durata e aggiungi a self.log
        duration = time.time() - self._start_time
        self.log.append({
            "tool":     self._current_tool,
            "duration": round(duration, 3),
            "timestamp": datetime.datetime.now().isoformat()
        })

    def report(self):
        print("\n📊 Tool Execution Log:")
        for entry in self.log:
            print(f"  {entry['tool']:35s} {entry['duration']:.3f}s")


# Decommenta per testare con entrambi i callback:
# TICKET_STORE.clear()
# tools_v2 = [get_employee_budget, search_product_catalog, request_purchase_approval]
# agent_v2  = create_tool_calling_agent(llm, tools_v2, prompt_v2)
# approval  = HumanApprovalCallback(auto_mode=True)
# logger    = ToolLogger()
# exec_dual = AgentExecutor(agent=agent_v2, tools=tools_v2,
#                           callbacks=[approval, logger], verbose=False)
# exec_dual.invoke({"input": "Giulia Verdi E1082, voglio il monitor 4K."})
# logger.report()
print("💡 Decommenta il blocco sopra per testare il dual-callback.")

💡 Decommenta il blocco sopra per testare il dual-callback.


---
### ✅ Recap Lab 3

- Abbiamo aggiunto `request_purchase_approval` come `@tool` per l'escalation.
- Abbiamo implementato `HumanApprovalCallback(BaseCallbackHandler)` che intercetta l'evento `on_tool_end`.
- Il callback è **iniettato** nell'`AgentExecutor` tramite `callbacks=[...]` — nessuna modifica al loop.
- Abbiamo costruito un **audit trail** che registra ogni decisione umana.
- L'`auto_mode` permette di testare il flow senza input interattivo.

➡️ **Prossimo notebook:** `04_Agente_as_Tool.ipynb` — usare un `AgentExecutor` come tool di un altro agente.